In [4]:
import numpy as np
import cvxpy as cp

def random_spd(d, condition=10.0, rng=None):
    """Generate a random d x d SPD matrix with controlled spectrum."""
    rng = np.random.default_rng(rng)
    U, _ = np.linalg.qr(rng.normal(size=(d, d)))
    eigs = np.exp(np.linspace(0, np.log(condition), d))
    rng.shuffle(eigs)
    return U @ np.diag(eigs) @ U.T


def sqrtm_spd(S):
    """Symmetric square root of an SPD matrix."""
    eigvals, eigvecs = np.linalg.eigh(S)
    eigvals = np.maximum(eigvals, 0.0)
    return eigvecs @ np.diag(np.sqrt(eigvals)) @ eigvecs.T


def solve_Q_star(Sigma0, Sigma1, delta=None, solver="CLARABEL", verbose=False):
    """
    Solve the Gaussian Q-problem for gamma(S)=lambda_max(S):

        max_{Q >= 0, tr(Q)=1}
            delta^T Q delta
            + tr(Q Sigma0) + tr(Q Sigma1)
            - 2 || Sigma1^{1/2} Q Sigma0^{1/2} ||_*

    Returns Q_star, objective_value, eigenvalues of Q_star.
    """
    d = Sigma0.shape[0]
    if delta is None:
        delta = np.zeros(d)

    S0h = sqrtm_spd(Sigma0)
    S1h = sqrtm_spd(Sigma1)

    Q = cp.Variable((d, d), symmetric=True)

    linear_part = (
        cp.quad_form(delta, Q)
        + cp.trace(Q @ Sigma0)
        + cp.trace(Q @ Sigma1)
    )

    # Nuclear norm term. This is convex in Q, so -2 * normNuc is concave.
    nuclear_part = cp.normNuc(S1h @ Q @ S0h)

    objective = cp.Maximize(linear_part - 2 * nuclear_part)

    constraints = [
        Q >> 0,
        cp.trace(Q) == 1,
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=solver, verbose=verbose)

    Q_star = Q.value
    Q_star = 0.5 * (Q_star + Q_star.T)

    eigvals = np.linalg.eigvalsh(Q_star)
    eigvals = np.sort(eigvals)[::-1]

    return Q_star, problem.value, eigvals


def numerical_rank(eigvals, tol=1e-4):
    return np.sum(eigvals > tol)


def experiment(d=5, n_trials=20, condition=10.0, with_mean=False, seed=0):
    rng = np.random.default_rng(seed)

    ranks = []
    eigs_all = []

    for t in range(n_trials):
        Sigma0 = random_spd(d, condition=condition, rng=rng)
        Sigma1 = random_spd(d, condition=condition, rng=rng)

        if with_mean:
            delta = rng.normal(size=d)
        else:
            delta = np.zeros(d)

        Q_star, value, eigvals = solve_Q_star(Sigma0, Sigma1, delta)

        rank = numerical_rank(eigvals)
        ranks.append(rank)
        eigs_all.append(eigvals)

        print(f"Trial {t+1:02d}: rank(Q*) = {rank}, eigvals = {eigvals}")

    ranks = np.array(ranks)

    print("\nSummary")
    print("-------")
    print(f"dimension: {d}")
    print(f"trials: {n_trials}")
    print(f"with_mean: {with_mean}")
    print(f"rank-one frequency: {np.mean(ranks == 1):.2%}")
    print(f"average numerical rank: {np.mean(ranks):.2f}")

    return ranks, np.array(eigs_all)


In [6]:
ranks, eigs = experiment(
    d=5,
    n_trials=30,
    condition=20.0,
    with_mean=True,
    seed=123,
    )

Trial 01: rank(Q*) = 1, eigvals = [ 9.99999955e-01  2.92443077e-08  1.61774339e-08  8.46308172e-09
 -8.80749964e-09]
Trial 02: rank(Q*) = 1, eigvals = [ 9.99999993e-01  4.83303933e-09  1.81535700e-09  3.13806749e-10
 -3.36124591e-10]
Trial 03: rank(Q*) = 1, eigvals = [9.99999900e-01 8.98695255e-08 6.01230602e-09 4.10150685e-09
 4.03460529e-10]
Trial 04: rank(Q*) = 1, eigvals = [9.99999992e-01 4.21927979e-09 2.82267079e-09 1.02095235e-09
 2.79404331e-10]
Trial 05: rank(Q*) = 1, eigvals = [9.99999940e-01 4.35018918e-08 9.07326405e-09 5.49160322e-09
 1.65948808e-09]
Trial 06: rank(Q*) = 1, eigvals = [9.99999877e-01 7.99999589e-08 2.45348049e-08 1.18712257e-08
 6.76410371e-09]
Trial 07: rank(Q*) = 1, eigvals = [9.99999934e-01 5.53315929e-08 7.64414392e-09 2.20183479e-09
 3.38595989e-10]
Trial 08: rank(Q*) = 1, eigvals = [9.99999978e-01 1.33267389e-08 4.60149995e-09 3.28714455e-09
 4.77734419e-10]
Trial 09: rank(Q*) = 1, eigvals = [9.99999990e-01 6.67882449e-09 2.11695411e-09 6.79746544e-10